In [80]:
from pyspark.sql import SparkSession
import re
from pyspark.sql.types import DoubleType, IntegerType, DateType


path_raw = 'files/sales_raw.txt'
path_final_csv = 'files/sales_final.csv'


## Creates the spark session

In [81]:
spark = SparkSession.builder \
    .appName("SalesApp") \
    .getOrCreate()

## creates the empty list

In [221]:

list_sales = []

## read the lines

In [222]:

file = open(path_raw, 'r')
for line in file:
    dict_sales = {}
    dict_sales['transaction_id'] = re.search(r'TRANSACTION_ID: (.*?)(?=\|)', line).group(1).strip()
    dict_sales['date'] = re.search(r'DATE: (.*?)(?=\|)', line).group(1).strip()
    dict_sales['customer'] = re.search(r'CUSTOMER: (.*?)(?=\|)', line).group(1).strip()
    dict_sales['product'] = re.search(r'PRODUCT: (.*?)(?=\|)', line).group(1).strip()
    dict_sales['total'] = re.search(r'TOTAL: (.*?)(?=\|)', line).group(1).strip()
    dict_sales['status'] = re.search(r'STATUS: (.*)', line).group(1).strip()
    list_sales.append(dict_sales)
file.close()

In [223]:
print(list_sales)

[{'transaction_id': '1007', 'date': '2026-01-18', 'customer': 'Elena_Rius', 'product': 'Ergonomic_Chair', 'total': '210.75', 'status': 'Pending'}, {'transaction_id': '1007', 'date': '2026-01-18', 'customer': 'Elena_Rius', 'product': 'Ergonomic_Chair', 'total': '210.75', 'status': 'Pending'}, {'transaction_id': '1007', 'date': '2026-01-18', 'customer': 'Elena_Rius', 'product': 'Ergonomic_Chair', 'total': '210.75', 'status': 'Pending'}, {'transaction_id': '1008', 'date': '2026-01-19', 'customer': 'Juan_Perez', 'product': 'Laptop_Stand', 'total': '89.50', 'status': 'Completed'}, {'transaction_id': '1008', 'date': '2026-01-19', 'customer': 'Juan_Perez', 'product': 'Laptop_Stand', 'total': '89.50', 'status': 'Completed'}, {'transaction_id': '1009', 'date': '2026-01-20', 'customer': 'Maria_Lopez', 'product': 'Desk_Lamp', 'total': '45.00', 'status': 'Pending'}, {'transaction_id': '1010', 'date': '2026-01-20', 'customer': 'Maria_Lopez', 'product': 'Desk_Lamp', 'total': '45.00', 'status': ''}, 

## create the spark Dataframe

In [224]:

df = spark.createDataFrame(list_sales)
df.show(1000)

+-----------------+----------+---------------+---------+------+--------------+
|         customer|      date|        product|   status| total|transaction_id|
+-----------------+----------+---------------+---------+------+--------------+
|       Elena_Rius|2026-01-18|Ergonomic_Chair|  Pending|210.75|          1007|
|       Elena_Rius|2026-01-18|Ergonomic_Chair|  Pending|210.75|          1007|
|       Elena_Rius|2026-01-18|Ergonomic_Chair|  Pending|210.75|          1007|
|       Juan_Perez|2026-01-19|   Laptop_Stand|Completed| 89.50|          1008|
|       Juan_Perez|2026-01-19|   Laptop_Stand|Completed| 89.50|          1008|
|      Maria_Lopez|2026-01-20|      Desk_Lamp|  Pending| 45.00|          1009|
|      Maria_Lopez|2026-01-20|      Desk_Lamp|         | 45.00|          1010|
|   Carlos_Sanchez|2026-01-21|    Monitor_Arm|  Shipped|120.99|          1011|
|   Carlos_Sanchez|2026-01-21|    Monitor_Arm|  Shipped|      |          1012|
|        Ana_Gomez|2026-01-22|       Keyboard|Cancel

In [225]:
df.printSchema()

root
 |-- customer: string (nullable = true)
 |-- date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total: string (nullable = true)
 |-- transaction_id: string (nullable = true)



## transform data

In [226]:
df = df.replace("", None)

In [227]:
df.show()

+-----------------+----------+---------------+---------+------+--------------+
|         customer|      date|        product|   status| total|transaction_id|
+-----------------+----------+---------------+---------+------+--------------+
|       Elena_Rius|2026-01-18|Ergonomic_Chair|  Pending|210.75|          1007|
|       Elena_Rius|2026-01-18|Ergonomic_Chair|  Pending|210.75|          1007|
|       Elena_Rius|2026-01-18|Ergonomic_Chair|  Pending|210.75|          1007|
|       Juan_Perez|2026-01-19|   Laptop_Stand|Completed| 89.50|          1008|
|       Juan_Perez|2026-01-19|   Laptop_Stand|Completed| 89.50|          1008|
|      Maria_Lopez|2026-01-20|      Desk_Lamp|  Pending| 45.00|          1009|
|      Maria_Lopez|2026-01-20|      Desk_Lamp|     NULL| 45.00|          1010|
|   Carlos_Sanchez|2026-01-21|    Monitor_Arm|  Shipped|120.99|          1011|
|   Carlos_Sanchez|2026-01-21|    Monitor_Arm|  Shipped|  NULL|          1012|
|        Ana_Gomez|2026-01-22|       Keyboard|Cancel

In [228]:

df = (
    df
    .withColumn("date", df["date"].cast(DateType()))
    .withColumn("total", df["total"].cast(DoubleType()))
    .withColumn("transaction_id", df["transaction_id"].cast(IntegerType()))
    )

In [229]:
df.printSchema()

root
 |-- customer: string (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total: double (nullable = true)
 |-- transaction_id: integer (nullable = true)



## Filtering columns

In [230]:
df.select('customer', 'product').show(100)

+-----------------+---------------+
|         customer|        product|
+-----------------+---------------+
|       Elena_Rius|Ergonomic_Chair|
|       Elena_Rius|Ergonomic_Chair|
|       Elena_Rius|Ergonomic_Chair|
|       Juan_Perez|   Laptop_Stand|
|       Juan_Perez|   Laptop_Stand|
|      Maria_Lopez|      Desk_Lamp|
|      Maria_Lopez|      Desk_Lamp|
|   Carlos_Sanchez|    Monitor_Arm|
|   Carlos_Sanchez|    Monitor_Arm|
|        Ana_Gomez|       Keyboard|
|        Ana_Gomez|       Keyboard|
|     Luis_Ramirez|          Mouse|
|     Luis_Ramirez|          Mouse|
|     Luis_Ramirez|          Mouse|
|     Sofia_Martin|  Chair_Cushion|
|     Sofia_Martin|  Chair_Cushion|
|Roberto_Fernandez| Desk_Organizer|
|Roberto_Fernandez| Desk_Organizer|
|Roberto_Fernandez| Desk_Organizer|
| Gabriela_Morales|     Lamp_Shade|
| Gabriela_Morales|     Lamp_Shade|
|    Miguel_Torres|       Notebook|
|    Miguel_Torres|       Notebook|
|    Miguel_Torres|       Notebook|
|     Isabel_Ortiz|        P

In [231]:
df.select('customer', 'product').filter(df.customer == 'Ana_Gomez').show()

+---------+--------+
| customer| product|
+---------+--------+
|Ana_Gomez|Keyboard|
|Ana_Gomez|Keyboard|
+---------+--------+



## Cleans the _ symbols

In [232]:
from pyspark.sql.functions import regexp_replace

df = (
        df
            .withColumn("customer", regexp_replace("customer", r"[_]", " "))
            .withColumn("product", regexp_replace("product", r"[_]", " "))
    )

In [233]:
df.select('customer', 'product').show()

+-----------------+---------------+
|         customer|        product|
+-----------------+---------------+
|       Elena Rius|Ergonomic Chair|
|       Elena Rius|Ergonomic Chair|
|       Elena Rius|Ergonomic Chair|
|       Juan Perez|   Laptop Stand|
|       Juan Perez|   Laptop Stand|
|      Maria Lopez|      Desk Lamp|
|      Maria Lopez|      Desk Lamp|
|   Carlos Sanchez|    Monitor Arm|
|   Carlos Sanchez|    Monitor Arm|
|        Ana Gomez|       Keyboard|
|        Ana Gomez|       Keyboard|
|     Luis Ramirez|          Mouse|
|     Luis Ramirez|          Mouse|
|     Luis Ramirez|          Mouse|
|     Sofia Martin|  Chair Cushion|
|     Sofia Martin|  Chair Cushion|
|Roberto Fernandez| Desk Organizer|
|Roberto Fernandez| Desk Organizer|
|Roberto Fernandez| Desk Organizer|
| Gabriela Morales|     Lamp Shade|
+-----------------+---------------+
only showing top 20 rows


# DUPLICATES

### Show duplicates

In [234]:
from pyspark.sql.functions import count, col

df.groupBy('transaction_id').count().show(truncate=False)


+--------------+-----+
|transaction_id|count|
+--------------+-----+
|1008          |2    |
|1007          |3    |
|1010          |1    |
|1011          |1    |
|1009          |1    |
|1014          |2    |
|1012          |1    |
|1013          |2    |
|1016          |2    |
|1015          |1    |
|1019          |1    |
|1017          |2    |
|1018          |1    |
|1021          |2    |
|1022          |1    |
|1020          |1    |
|1025          |2    |
|1023          |2    |
|1024          |1    |
|1026          |1    |
+--------------+-----+
only showing top 20 rows


### Drop Duplicates

In [235]:
df_no_duplicates = df.dropDuplicates()

In [236]:
df_no_duplicates.show() # without duplicates

+-----------------+----------+---------------+---------+------+--------------+
|         customer|      date|        product|   status| total|transaction_id|
+-----------------+----------+---------------+---------+------+--------------+
|       Elena Rius|2026-01-18|Ergonomic Chair|  Pending|210.75|          1007|
|       Juan Perez|2026-01-19|   Laptop Stand|Completed|  89.5|          1008|
|   Carlos Sanchez|2026-01-21|    Monitor Arm|  Shipped|120.99|          1011|
|      Maria Lopez|2026-01-20|      Desk Lamp|  Pending|  45.0|          1009|
|        Ana Gomez|2026-01-22|       Keyboard|Cancelled|  35.5|          1013|
|     Luis Ramirez|2026-01-23|          Mouse|Completed|  25.0|          1014|
|     Sofia Martin|2026-01-24|  Chair Cushion|  Pending| 15.75|          1016|
|     Luis Ramirez|2026-01-23|          Mouse|Completed|  25.0|          1015|
| Gabriela Morales|2026-01-26|     Lamp Shade|  Shipped|  22.9|          1019|
|Roberto Fernandez|2026-01-25| Desk Organizer|Comple

In [237]:
df.count() # with duplicates

66

In [238]:
df_no_duplicates.count() # without duplicates

45

In [239]:
df = df_no_duplicates

## Count null values per column

In [240]:
from pyspark.sql.functions import col, sum, when

df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in df.columns]).show()

+--------+----+-------+------+-----+--------------+
|customer|date|product|status|total|transaction_id|
+--------+----+-------+------+-----+--------------+
|       0|   0|      0|     5|    4|             0|
+--------+----+-------+------+-----+--------------+



In [241]:
df.dtypes

[('customer', 'string'),
 ('date', 'date'),
 ('product', 'string'),
 ('status', 'string'),
 ('total', 'double'),
 ('transaction_id', 'int')]

In [242]:
df.describe().show()

+-------+-----------------+---------+---------+------------------+------------------+
|summary|         customer|  product|   status|             total|    transaction_id|
+-------+-----------------+---------+---------+------------------+------------------+
|  count|               45|       45|       40|                41|                45|
|   mean|             NULL|     NULL|     NULL| 54.40829268292683|            1029.0|
| stddev|             NULL|     NULL|     NULL|55.248604231348644|13.133925536563696|
|    min|        Ana Gomez|Bookshelf|Cancelled|            -12.75|              1007|
|    max|Valeria Hernandez|  Pen Set|  Shipped|            210.75|              1051|
+-------+-----------------+---------+---------+------------------+------------------+



## Drop nulls

In [243]:
# show data values nulls in status or total column
df.filter(df.status.isNull() | df.total.isNull()).show()

+-----------------+----------+--------------+-------+-----+--------------+
|         customer|      date|       product| status|total|transaction_id|
+-----------------+----------+--------------+-------+-----+--------------+
|      Maria Lopez|2026-01-20|     Desk Lamp|   NULL| 45.0|          1010|
|   Carlos Sanchez|2026-01-21|   Monitor Arm|Shipped| NULL|          1012|
|Roberto Fernandez|2026-01-25|Desk Organizer|   NULL| 30.5|          1018|
|    Miguel Torres|2026-01-27|      Notebook|Pending| NULL|          1022|
|       Carla Vega|2026-01-30|     Bookshelf|   NULL| 85.0|          1029|
|   Esteban Moreno|2026-02-02|   Monitor Arm|Shipped| NULL|          1035|
|     Natalia Cruz|2026-02-05|      Notebook|Pending| NULL|          1041|
|Valeria Hernandez|2026-02-07|Desk Organizer|   NULL| 30.5|          1045|
|     Claudia Soto|2026-02-09|     Bookshelf|   NULL| 85.0|          1049|
+-----------------+----------+--------------+-------+-----+--------------+



In [244]:
df = df.dropna()

In [245]:
df.count()

36

## Count transactions by customer

In [246]:
df.groupBy('Customer').count().show(100)

+-----------------+-----+
|         Customer|count|
+-----------------+-----+
|     Luis Ramirez|    2|
|   Fernando Rojas|    2|
|Roberto Fernandez|    1|
|   Esteban Moreno|    1|
|Santiago Martinez|    2|
|    Paula Ramirez|    2|
| Gabriela Morales|    2|
|     Natalia Cruz|    1|
|   Carlos Sanchez|    1|
|     Ricardo Luna|    2|
|       Carla Vega|    2|
|       Juan Perez|    1|
|     Claudia Soto|    1|
|        Ana Gomez|    1|
|    Diego Alvarez|    2|
|       Elena Rius|    1|
|     Isabel Ortiz|    2|
|    Miguel Torres|    1|
|      Maria Lopez|    1|
|   Lorena Paredes|    2|
|     Sofia Martin|    1|
|Valeria Hernandez|    1|
|   Antonio Mendez|    2|
|    Javier Castro|    2|
+-----------------+-----+



## Check Ids with duplicates


We don't have duplicates because we have deleted them before

In [247]:
df.groupby('transaction_id').count().show(1000)

+--------------+-----+
|transaction_id|count|
+--------------+-----+
|          1025|    1|
|          1016|    1|
|          1031|    1|
|          1051|    1|
|          1030|    1|
|          1034|    1|
|          1019|    1|
|          1046|    1|
|          1008|    1|
|          1047|    1|
|          1021|    1|
|          1026|    1|
|          1028|    1|
|          1032|    1|
|          1048|    1|
|          1050|    1|
|          1017|    1|
|          1037|    1|
|          1036|    1|
|          1015|    1|
|          1020|    1|
|          1007|    1|
|          1039|    1|
|          1038|    1|
|          1042|    1|
|          1014|    1|
|          1027|    1|
|          1023|    1|
|          1043|    1|
|          1040|    1|
|          1024|    1|
|          1033|    1|
|          1011|    1|
|          1013|    1|
|          1044|    1|
|          1009|    1|
+--------------+-----+



## Negative values in total field

In [248]:
df.select('transaction_id','total').filter(df.total < 0).show(100)

+--------------+------+
|transaction_id| total|
+--------------+------+
|          1024|-12.75|
+--------------+------+



We have to decide if we delete this row

In [249]:
df = df.filter(df.total >= 0)

In [250]:
df.show(100)

+-----------------+----------+---------------+---------+------+--------------+
|         customer|      date|        product|   status| total|transaction_id|
+-----------------+----------+---------------+---------+------+--------------+
|       Elena Rius|2026-01-18|Ergonomic Chair|  Pending|210.75|          1007|
|       Juan Perez|2026-01-19|   Laptop Stand|Completed|  89.5|          1008|
|   Carlos Sanchez|2026-01-21|    Monitor Arm|  Shipped|120.99|          1011|
|      Maria Lopez|2026-01-20|      Desk Lamp|  Pending|  45.0|          1009|
|        Ana Gomez|2026-01-22|       Keyboard|Cancelled|  35.5|          1013|
|     Luis Ramirez|2026-01-23|          Mouse|Completed|  25.0|          1014|
|     Sofia Martin|2026-01-24|  Chair Cushion|  Pending| 15.75|          1016|
|     Luis Ramirez|2026-01-23|          Mouse|Completed|  25.0|          1015|
| Gabriela Morales|2026-01-26|     Lamp Shade|  Shipped|  22.9|          1019|
|Roberto Fernandez|2026-01-25| Desk Organizer|Comple

## Check dates

In [251]:
df.select('transaction_id','date').orderBy('date').show(1000)

+--------------+----------+
|transaction_id|      date|
+--------------+----------+
|          1007|2026-01-18|
|          1008|2026-01-19|
|          1009|2026-01-20|
|          1011|2026-01-21|
|          1013|2026-01-22|
|          1014|2026-01-23|
|          1015|2026-01-23|
|          1016|2026-01-24|
|          1017|2026-01-25|
|          1019|2026-01-26|
|          1020|2026-01-26|
|          1021|2026-01-27|
|          1023|2026-01-28|
|          1025|2026-01-29|
|          1026|2026-01-29|
|          1028|2026-01-30|
|          1027|2026-01-30|
|          1030|2026-01-31|
|          1031|2026-01-31|
|          1033|2026-02-01|
|          1032|2026-02-01|
|          1034|2026-02-02|
|          1036|2026-02-03|
|          1037|2026-02-03|
|          1038|2026-02-04|
|          1039|2026-02-04|
|          1040|2026-02-05|
|          1042|2026-02-06|
|          1043|2026-02-06|
|          1044|2026-02-07|
|          1047|2026-02-08|
|          1046|2026-02-08|
|          1048|2026

## Dataframe to csv file

In [252]:

df.coalesce(1).write.csv(path_final_csv, header=True, mode="overwrite")


In [259]:
df_csv = spark.read.option("header", True) \
               .option("inferSchema", True) \
               .csv(path_final_csv)

In [260]:
df_csv.show()

+-----------------+----------+---------------+---------+------+--------------+
|         customer|      date|        product|   status| total|transaction_id|
+-----------------+----------+---------------+---------+------+--------------+
|       Elena Rius|2026-01-18|Ergonomic Chair|  Pending|210.75|          1007|
|       Juan Perez|2026-01-19|   Laptop Stand|Completed|  89.5|          1008|
|   Carlos Sanchez|2026-01-21|    Monitor Arm|  Shipped|120.99|          1011|
|      Maria Lopez|2026-01-20|      Desk Lamp|  Pending|  45.0|          1009|
|        Ana Gomez|2026-01-22|       Keyboard|Cancelled|  35.5|          1013|
|     Luis Ramirez|2026-01-23|          Mouse|Completed|  25.0|          1014|
|     Sofia Martin|2026-01-24|  Chair Cushion|  Pending| 15.75|          1016|
|     Luis Ramirez|2026-01-23|          Mouse|Completed|  25.0|          1015|
| Gabriela Morales|2026-01-26|     Lamp Shade|  Shipped|  22.9|          1019|
|Roberto Fernandez|2026-01-25| Desk Organizer|Comple